# בניית מודל לזיהוי נטישת לקוחות (Churn) עם Scikit-Learn
### תרגיל מודרך שלב-אחר-שלב

**התרחיש:** את/ה מנהל/ת רשת סלולר. מדי חודש מיליוני לקוחות משלמים חשבון, מתקשרים
ומשתמשים בחבילת הגלישה שלהם. אבל קבוצה שקטה מהם עוברת ("נוטשת") למתחרה, נותנת
לחוזה לפוג, או פשוט מפסיקה לענות לשיחות החידוש.

עלות גיוס לקוח חדש (פרסום, סבסוד מכשירים, עמלות) גבוהה פי 5–7 מעלות שימור לקוח
קיים. נקודת אחוז אחת של שיפור בשימור לקוחות יכולה להיות שווה עשרות מיליוני
דולרים בשנה עבור חברת סלולר בינונית. אם אפשר לחזות מי עומד לעזוב **לפני** שהוא
עוזב בפועל — אפשר להתערב: להציע תעריף טוב יותר, לזרז טיפול בתלונה, או לסמן את
החשבון לשיחה יזומה מצוות השימור.

בתרגיל הזה נבנה, שלב אחרי שלב, מודל שמזהה אילו לקוחות בסיכון לנטישה — באמצעות
הספרייה **Scikit-Learn**, ספריית למידת המכונה הפופולרית ביותר בפייתון.

---

## מטרות לימוד

1. להבין את ה־**Estimator API** של Scikit-Learn — הדפוס האחיד שחוזר על עצמו בכל
   מודל בספרייה, מרגרסיה לינארית ועד יער אקראי.
2. להבין את ייצוג הנתונים הסטנדרטי: **מטריצת Features** (`X`) ו־**וקטור Target** (`y`).
3. לבנות פייפליין מלא: יצירת נתונים → EDA → עיבוד מקדים → אימון → הערכה → כיוונון → פרשנות.
4. להבין למה **דיוק (accuracy)** הוא מדד מטעה כשיש חוסר איזון בין המחלקות, ואיזה
   מדדים כן משקפים ביצועים אמיתיים.


## שלב 0: מבוא ל־Scikit-Learn — ה־Estimator API

הרעיון המרכזי של Scikit-Learn הוא שלכל מודל למידת מכונה — בין אם זו רגרסיה
לינארית, יער אקראי, או רשת עצבית פשוטה — יש **אותו ממשק בדיוק**. ברגע שלמדת
לעבוד עם מודל אחד, ידעת לעבוד עם כולם.

התהליך תמיד נראה כך:

| שלב | מה עושים | דוגמה |
|---|---|---|
| 1. בחירת מחלקת מודל | מייבאים את סוג האסטימטור | `from sklearn.ensemble import RandomForestClassifier` |
| 2. בחירת היפרפרמטרים | יוצרים מופע (instance) עם הגדרות | `model = RandomForestClassifier(n_estimators=200)` |
| 3. סידור הנתונים | בונים מטריצת `X` ווקטור `y` | `X = df[features]`, `y = df['churn']` |
| 4. אימון (`fit`) | המודל "לומד" מהנתונים | `model.fit(X_train, y_train)` |
| 5. חיזוי (`predict`) | מפעילים על נתונים חדשים | `model.predict(X_test)` |

יש גם משפחה שנייה של אובייקטים — **Transformers** (כמו `StandardScaler` או
`LabelEncoder`) — שמשתמשים ב־`fit` / `transform` במקום `fit` / `predict`, כי
התפקיד שלהם הוא לשנות את הנתונים, לא לחזות מהם תווית.

נראה את שני הדפוסים האלה בפעולה לאורך התרגיל.


## שלב 1: ייצוג הנתונים ב־Scikit-Learn

כל מודל ב־Scikit-Learn מצפה לנתונים בפורמט קבוע:

- **`X` — מטריצת Features**: טבלה דו-ממדית בגודל `(n_samples, n_features)`.
  כל שורה היא תצפית (במקרה שלנו: לקוח), כל עמודה היא מאפיין (tenure, חיובים
  חודשיים, שיחות לשירות לקוחות...).
- **`y` — וקטור Target**: מערך חד-ממדי באורך `n_samples` עם התווית שרוצים לחזות.
  אצלנו: `0` = לקוח פעיל, `1` = לקוח שנטש.

זו הבעיה שלנו במונחים סטטיסטיים: **סיווג בינארי (binary classification)**.


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve)

np.random.seed(42)

## שלב 2: יצירת דאטהסט סינתטי

נדמה מפעיל סלולר עם 3,333 לקוחות ושיעור נטישה ריאליסטי של כ-14.3%. תווית
הנטישה נוצרת ממודל לוגיסטי, כך שיותר שיחות לשירות לקוחות, חבילת נדידה
בינלאומית, חיוב חודשי גבוה, וותק קצר — כולם מעלים את הסתברות הנטישה.

בפרויקט אמיתי הנתונים היו מגיעים ממחסן נתונים (data warehouse) של החברה; כאן
אנחנו מייצרים אותם כדי שהתרגיל יהיה ניתן לשחזור ולהרצה על ידי כל סטודנט/ית.


In [2]:
n = 3333

df = pd.DataFrame({
    'tenure':                  np.random.randint(1, 72, n),
    'monthly_charges':         np.round(np.random.uniform(18, 119, n), 2),
    'total_charges':           np.round(np.random.uniform(18, 8700, n), 2),
    'customer_service_calls':  np.random.randint(0, 10, n),
    'day_mins':                np.round(np.random.normal(180, 55, n).clip(0), 1),
    'day_calls':               np.random.randint(50, 165, n),
    'intl_mins':               np.round(np.random.normal(10, 4, n).clip(0), 1),
    'intl_plan':               np.random.choice(['yes', 'no'], n, p=[0.1, 0.9]),
    'voice_mail_plan':         np.random.choice(['yes', 'no'], n, p=[0.28, 0.72]),
    'number_vmail_msgs':       np.random.randint(0, 52, n),
})

# אות נטישה לוגיסטי: שיחות שירות, חבילה בינלאומית, חיובים, וותק קצר מגבירים נטישה
logit = (
    -4.2
    + 0.35 * df['customer_service_calls']
    + 2.20 * (df['intl_plan'] == 'yes').astype(float)
    + 0.015 * df['monthly_charges']
    - 0.030 * df['tenure']
    + np.random.normal(0, 0.5, n)
)
churn_prob = 1 / (1 + np.exp(-logit))
df['churn'] = (np.random.uniform(0, 1, n) < churn_prob).astype(int)

print(df.shape)
print(df['churn'].value_counts())
df.head(3)

(3333, 11)
churn
0    2857
1     476
Name: count, dtype: int64


,tenure,monthly_charges,total_charges,customer_service_calls,day_mins,day_calls,intl_mins,intl_plan,voice_mail_plan,number_vmail_msgs,churn
0,52,61.97,6123.18,6,205.7,79,13.3,no,no,11,0
1,15,73.48,206.40,9,150.8,116,10.3,no,yes,19,1
2,61,49.51,5497.97,2,229.5,100,6.6,no,no,16,0


**מה רואים כאן?** 3,333 לקוחות, מתוכם 476 נטשו (כ-14.3%). זהו חוסר איזון
טיפוסי בין מחלקות (class imbalance) — הרבה יותר לקוחות נאמנים מנוטשים. מודל
שתמיד יחזה "לא נטש" יקבל דיוק (accuracy) של 85.7% בלי שיהיה שימושי בכלל.
נזכור את זה כשנגיע לשלב ההערכה.

## שלב 3: ניתוח נתונים אקספלורטיבי (EDA)

לפני שבונים מודל, כדאי לבדוק איך שיעור הנטישה משתנה לפי המאפיינים החשובים
ביותר. זה נותן לנו אינטואיציה על אילו סיגנלים צפויים להיות משמעותיים, ומשמש
בדיקת שפיות (sanity check) להתנהגות המודל בהמשך.


In [3]:
cs_churn = df.groupby('customer_service_calls')['churn'].mean().reset_index()
cs_churn.columns = ['calls', 'churn_rate']

intl_churn = df.groupby('intl_plan')['churn'].agg(['mean', 'count'])
intl_churn.columns = ['churn_rate', 'count']

print("=== שיעור נטישה לפי מספר שיחות לשירות לקוחות ===")
print(cs_churn.to_string(index=False))

print("\n=== שיעור נטישה לפי חבילה בינלאומית ===")
print(intl_churn)

=== שיעור נטישה לפי מספר שיחות לשירות לקוחות ===
 calls  churn_rate
     0    0.048485
     1    0.034783
     2    0.062305
     3    0.078873
     4    0.120482
     5    0.123418
     6    0.148571
     7    0.211480
     8    0.280374
     9    0.328313

=== שיעור נטישה לפי חבילה בינלאומית ===
           churn_rate  count
intl_plan                   
no           0.107286   2992
yes          0.454545    341


**שני סיגנלים ברורים עולים:** שיעור הנטישה עולה בחדות עם מספר שיחות
השירות — לקוחות עם 7+ שיחות נוטשים בשיעור של מעל 21%, לעומת כ-5% אצל מי שלא
התקשר בכלל. בעלי חבילה בינלאומית נוטשים ב-45.5% לעומת 10.7% אצל השאר — תבנית
מוכרת מאוד בתחום הסלולר, שבה חיובי נדידה מייצרים "הלם חשבון" ותלונות. שני
המאפיינים האלה יהיו קריטיים במודל שלנו.

## שלב 4: עיבוד מקדים — Transformers ב־Scikit-Learn

עכשיו ניגע במשפחה השנייה של אובייקטים ב־Scikit-Learn: **Transformers**. בניגוד
ל־Estimator (`fit` + `predict`), ל-Transformer יש `fit` + `transform`, כי
תפקידו לשנות את הנתונים עצמם.

- **`LabelEncoder`** הופך עמודות טקסט (`'yes'`/`'no'`) למספרים (`0`/`1`).
- **`StandardScaler`** מנרמל כל מאפיין מספרי לממוצע 0 וסטיית תקן 1.

אנחנו גם מבצעים כאן חלוקה מדורגת (**stratified split**) ל-80% אימון ו-20%
בדיקה, כדי לשמר את יחס 14.3% הנטישה בשני החלקים. בלי זה, ייתכן במקרה שכל
הנוטשים ייפלו לחלק אחד בלבד.

**⚠️ נקודה קריטית ללימוד — דליפת מידע (Data Leakage):** תמיד `fit` על נתוני
האימון בלבד, ו-`transform` בלבד (לא `fit`) על נתוני הבדיקה. אם ה־Scaler
"רואה" את נתוני הבדיקה בזמן ה-`fit`, המודל מקבל מידע סטטיסטי על הבדיקה עוד
לפני שהוערך עליה — וזה מנפח באופן מלאכותי את תוצאות ההערכה.


In [4]:
le = LabelEncoder()
df['intl_plan_enc']  = le.fit_transform(df['intl_plan'])       # no=0, yes=1
df['voice_plan_enc'] = le.fit_transform(df['voice_mail_plan']) # no=0, yes=1

features = [
    'tenure', 'monthly_charges', 'total_charges',
    'customer_service_calls', 'day_mins', 'day_calls',
    'intl_mins', 'number_vmail_msgs',
    'intl_plan_enc', 'voice_plan_enc'
]
X = df[features]
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=6, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)   # fit + transform על האימון
X_test_sc  = scaler.transform(X_test)        # transform בלבד על הבדיקה!

print(f"אימון: {X_train.shape[0]} דוגמאות | בדיקה: {X_test.shape[0]} דוגמאות")
print(f"שיעור נטישה באימון: {y_train.mean():.3f}")
print(f"שיעור נטישה בבדיקה: {y_test.mean():.3f}")

אימון: 2666 דוגמאות | בדיקה: 667 דוגמאות
שיעור נטישה באימון: 0.143
שיעור נטישה בבדיקה: 0.142


שני הפיצולים שומרים על שיעור נטישה של כ-14.3%, סימן שהחלוקה המדורגת
עובדת כמצופה.

## שלב 5: אימון מודל — ה־Estimator API בפעולה

הגיע הזמן ליישם בפועל את חמשת השלבים שראינו בשלב 0. נבחר ב־**Random Forest**
כמודל ראשון: הוא מטפל היטב ביחסים לא-לינאריים, עובד טוב עם מאפיינים מעורבים,
ופחות רגיש לבחירת היפרפרמטרים מרוב המודלים האחרים.

נשתמש ב-`class_weight='balanced'` כדי לפצות על חוסר האיזון בין המחלקות —
המודל "מעניש" את עצמו יותר על טעות בזיהוי לקוח שנטש, כי מחלקת הנטישה נדירה יותר.

לפני האימון הסופי נריץ **Cross-Validation**: מחלקים את נתוני האימון ל-5
חלקים (folds), מאמנים על 4 ומאמתים על ה-5, וחוזרים 5 פעמים כך שכל חלק משמש
פעם אחת לאימות. כך מקבלים 5 הערכות בלתי-תלויות בלי דליפת מידע — תמונה הרבה
יותר אמינה מפיצול יחיד.


In [5]:
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    class_weight='balanced',   # מגדיל את המשקל של מחלקת הנטישה המיעוטית
    random_state=42,
    n_jobs=-1
)

# Cross-validation על נתוני האימון (5 folds, מדד ROC-AUC)
cv_scores = cross_val_score(rf, X_train_sc, y_train, cv=5, scoring='roc_auc')

rf.fit(X_train_sc, y_train)   # <-- זהו! זה כל האימון: fit(X, y)

print(f"ציוני CV ROC-AUC: {cv_scores.round(3)}")
print(f"ממוצע CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

ציוני CV ROC-AUC: [0.802 0.8   0.79  0.82  0.757]
ממוצע CV AUC: 0.7939 ± 0.0210


ממוצע CV AUC סביב 0.79 עם שונות נמוכה הוא תוצאה סבירה מאוד לדאטהסט
טלקום רועש. AUC מודד את יכולת המודל לדרג נוטשים מעל לא-נוטשים: ציון של 0.79
אומר שב-79% מהמקרים המודל נותן הסתברות נטישה גבוהה יותר ללקוח שאכן נטש
לעומת לקוח אקראי שלא נטש. ציונים עקביים על פני 5 ה-folds מעידים על מודל
יציב שלא "משנן" פלח מסוים של הנתונים.

## שלב 6: הערכת המודל על נתוני בדיקה

עכשיו נבדוק את המודל על 667 הלקוחות שהוא **מעולם לא ראה**. מטריצת הבלבול
(confusion matrix) היא הסיכום הכי כן של התנהגות מסווג: היא מראה בדיוק מה
המודל צדק ומה טעה, ולאיזה כיוון.


In [6]:
y_pred = rf.predict(X_test_sc)              # תווית חזויה: 0 או 1
y_prob = rf.predict_proba(X_test_sc)[:, 1]   # הסתברות נטישה: 0.0 עד 1.0

cm       = confusion_matrix(y_test, y_pred)
report   = classification_report(y_test, y_pred, target_names=['לא נטש', 'נטש'])
test_auc = roc_auc_score(y_test, y_prob)

print("מטריצת בלבול:\n", cm)
print("\nדוח סיווג:\n", report)
print(f"ROC-AUC על נתוני בדיקה: {test_auc:.4f}")

מטריצת בלבול:
 [[479  93]
 [ 39  56]]

דוח סיווג:
               precision    recall  f1-score   support

      לא נטש       0.92      0.84      0.88       572
         נטש       0.38      0.59      0.46        95

    accuracy                           0.80       667
   macro avg       0.65      0.71      0.67       667
weighted avg       0.85      0.80      0.82       667

ROC-AUC על נתוני בדיקה: 0.7940


**⚠️ מלכודת הדיוק (Accuracy Trap):** עם 14.3% נוטשים, מודל שתמיד חוזה
"לא נטש" יקבל 86% דיוק — חסר תועלת לחלוטין. במשימת נטישה מתעניינים בעיקר
ב-**Recall** (כמה נוטשים אמיתיים תפסנו) וב-**ROC-AUC** (יכולת דירוג כללית).

| מדד | נוסחה | מה הוא עונה |
|---|---|---|
| Precision | TP / (TP + FP) | מכל מי שסימנו כנוטש — כמה באמת נטשו? |
| Recall | TP / (TP + FN) | מכל הנוטשים האמיתיים — כמה תפסנו? |
| F1 | ממוצע הרמוני של Precision ו-Recall | איזון בין השניים |
| ROC-AUC | שטח מתחת לעקומת ה-ROC | דירוג נוטשים מעל לא-נוטשים על פני כל הספים |


## שלב 7: עקומת ROC וכיוונון סף ההחלטה

ברירת המחדל של Scikit-Learn מסווגת כ"נטש" כל הסתברות מעל 0.5. אבל 0.5 הוא
שרירותי — הוא לא נגזר מהעסק. **עקומת ה-ROC** מציגה את יחס True Positive Rate
מול False Positive Rate על פני **כל** הספים האפשריים, ועוזרת לבחור סף שמתאים
לצרכי העסק.

נשתמש בסטטיסטי **Youden's J** (`TPR - FPR`) כדי למצוא את הסף שממקסם את
ההפרש — כלומר את הנקודה שבה "מרוויחים הכי הרבה" מבחינת תפיסת נוטשים אמיתיים
תוך כמה שפחות אזעקות שווא.


In [7]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

j_scores    = tpr - fpr
optimal_idx = np.argmax(j_scores)
optimal_thr = thresholds[optimal_idx]

print(f"AUC = {test_auc:.4f}")
print(f"סף אופטימלי: {optimal_thr:.3f}")
print(f"בסף האופטימלי → TPR={tpr[optimal_idx]:.3f}, FPR={fpr[optimal_idx]:.3f}")

AUC = 0.7940
סף אופטימלי: 0.442
בסף האופטימלי → TPR=0.716, FPR=0.224


שימו לב: זו החלטה עסקית לא פחות משהיא סטטיסטית. הורדת הסף תופסת יותר
נוטשים אבל מוציאה תקציב שימור גם על לקוחות שלא היו עוזבים ממילא. העלאתו
מפחיתה אזעקות שווא אבל מפספסת יותר נוטשים אמיתיים. הבחירה תלויה בעלות
היחסית של נוטש שפוספס מול התערבות מבוזבזת.

## שלב 8: חשיבות מאפיינים (Feature Importance)

יער אקראי מספק לנו "בונוס" חשוב: לכל מאפיין הוא מחשב כמה הוא תרם להחלטות
המודל (Gini importance). זה משרת שתי מטרות: מוודא שהמודל למד תבניות הגיוניות,
ומספק תובנה עסקית — *למה* לקוחות נוטשים, לא רק *מי* עומד לנטוש.


In [8]:
importances = pd.Series(rf.feature_importances_, index=features)
importances = importances.sort_values(ascending=False)

print("=== חשיבות מאפיינים (Gini) ===")
for feat, val in importances.items():
    bar = '█' * int(val * 200)
    print(f"  {feat:<30} {val:.4f}  {bar}")

=== חשיבות מאפיינים (Gini) ===
  customer_service_calls         0.2329  ██████████████████████████████████████████████
  intl_plan_enc                  0.1340  ██████████████████████████
  monthly_charges                0.1328  ██████████████████████████
  tenure                         0.1173  ███████████████████████
  total_charges                  0.0855  █████████████████
  day_mins                       0.0807  ████████████████
  intl_mins                      0.0713  ██████████████
  day_calls                      0.0689  █████████████
  number_vmail_msgs              0.0676  █████████████
  voice_plan_enc                 0.0090  █


ארבעת המובילים — שיחות שירות לקוחות, חיוב חודשי, חבילה בינלאומית
וותק — תואמים בדיוק את מה שראינו ב-EDA ואת עשרות שנות מחקר נטישה בתחום
הסלולר. שימו לב ש-`voice_plan` מדורג אחרון: כלומר האם ללקוח יש חבילת תא
קולי כמעט ולא משפיע. **מודל מכויל היטב תמיד ייתן חשיבות מאפיינים שמתיישבת
עם הגיון התחום; אם לא — כדאי לחשוד בנתונים או בתוויות.**

## שלב 9: ניקוד לקוחות חדשים

השלב האחרון בפרודקשן: בהינתן קבוצת לקוחות חדשה, מפיקים ציון הסתברות נטישה
(0 עד 100%) ומדרג סיכון. זה מה שנשלח לצוות השימור לצורך התערבות.


In [9]:
sample = X_test.iloc[:10].copy()
sample['churn_prob']   = (y_prob[:10] * 100).round(1)
sample['actual_churn'] = y_test.values[:10]
sample['risk_tier'] = pd.cut(
    sample['churn_prob'],
    bins=[0, 25, 50, 75, 100],
    labels=['נמוך', 'בינוני', 'גבוה', 'קריטי']
)

display_cols = ['tenure', 'customer_service_calls', 'monthly_charges',
                'churn_prob', 'risk_tier', 'actual_churn']
sample[display_cols]

,tenure,customer_service_calls,monthly_charges,churn_prob,risk_tier,actual_churn
704,57,3,65.03,3.4,נמוך,0
1735,13,6,105.79,41.1,בינוני,0
367,26,5,57.47,15.0,נמוך,0
879,20,0,51.54,6.0,נמוך,0
96,44,4,27.30,26.0,בינוני,0
2138,53,2,107.37,8.3,נמוך,1
1578,6,6,79.09,52.1,גבוה,0
2495,33,4,90.38,26.3,בינוני,0
385,10,6,91.16,59.2,גבוה,1
2346,51,4,112.03,24.5,נמוך,1


## שלב 10: כוחו של ה־Estimator API האחיד

זו הנקודה הכי חשובה ללמד בהקשר של Scikit-Learn: **מכיוון שלכל מודל אותו
ממשק בדיוק (`fit` / `predict`), אפשר להחליף מודל בלי לשנות כלום אחר
בפייפליין.** נדגים זאת בכך שנחליף את ה-Random Forest ברגרסיה לוגיסטית —
מודל שונה לחלוטין מבחינה מתמטית — עם שינוי של שתי שורות קוד בלבד.


In [10]:
logreg = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
logreg.fit(X_train_sc, y_train)   # אותה שיטה fit(), מודל אחר לגמרי

logreg_prob = logreg.predict_proba(X_test_sc)[:, 1]
logreg_auc  = roc_auc_score(y_test, logreg_prob)

print(f"Random Forest  ROC-AUC: {test_auc:.4f}")
print(f"Logistic Regression ROC-AUC: {logreg_auc:.4f}")

Random Forest  ROC-AUC: 0.7940
Logistic Regression ROC-AUC: 0.8273


שני המודלים משתמשים באותו `X_train_sc` / `y_train`, אותו `fit()`,
ואותו `predict_proba()`. זה כל הקסם של ה-Estimator API: לומדים דפוס אחד,
ומיישמים אותו על עשרות אלגוריתמים שונים.

## סיכום

בנינו פייפליין מלא של סיווג בינארי עם Scikit-Learn:

1. ייצגנו את הבעיה כ־`X` (מטריצת features) ו־`y` (וקטור target).
2. חקרנו את הנתונים (EDA) לפני שבנינו מודל.
3. עיבדנו מקדים עם **Transformers** (`LabelEncoder`, `StandardScaler`), תוך
   הקפדה על `fit` על אימון בלבד למניעת דליפת מידע.
4. אימנו **Estimator** (`RandomForestClassifier`) עם `fit()`.
5. הערכנו עם Cross-Validation ומדדים שמתאימים לחוסר איזון (Recall, ROC-AUC —
   לא Accuracy).
6. כיוונו את סף ההחלטה לפי הצרכים העסקיים.
7. פירשנו את המודל דרך חשיבות מאפיינים.
8. הפקנו ציוני סיכון לפעולה בפועל.
9. ראינו שה-API האחיד מאפשר להחליף מודלים (Random Forest ↔ Logistic Regression)
   כמעט בלי לשנות קוד.

### תרגילים להמשך לסטודנטים

1. **החליפו מודל:** נסו `from sklearn.tree import DecisionTreeClassifier` או
   `from sklearn.svm import SVC` במקום Random Forest. איך ה-AUC משתנה?
2. **הנדסת מאפיינים:** הוסיפו מאפיין חדש `charges_per_minute = monthly_charges / day_mins`.
   האם הוא משפר את המודל? איפה הוא מדורג בחשיבות המאפיינים?
3. **כיוונון היפרפרמטרים:** נסו `GridSearchCV` כדי לחפש את `max_depth` ו-`n_estimators`
   האופטימליים.
4. **דמיינו את עקומת ה-ROC:** ציירו את `fpr` מול `tpr` עם `matplotlib` וסמנו
   את הנקודה האופטימלית.
5. **שאלה עסקית:** אם עלות שיחת שימור היא 20$ ועלות איבוד לקוח היא 400$,
   באיזה סף כדאי לבחור? נסו לחשב את התועלת הצפויה (expected value) בכמה
   ספים שונים.
